# AAI-540 Machine Learning Operations (MLOps) Final Team Project

- **Name**: Pros Loung
- **Assigment 3.1**: Feature Store - Exercise
- **Professor**: Mark Christenson
- **Course**: Machine Learning Operations (MLOps) (AAI-540-01)

# Instructions
To complete the assignment use Lab 3.1 as an example; build your own feature store and feature groups, and perform some simple feature engineering tasks detailed in the instructions below. 

# Get Started
Complete the lecture presentation and Lab 3.1 tutorial in SageMaker prior to attempting this assignment.

1. To start the assignment, log on to AWS Learner Labs and launch SageMaker Studio.
2. Use the AAI-540 GitHub to clone the lab notebook if you have not already done so.
3. Run the notebook instance and repeat the lab notebook steps for SageMaker feature store using the Homework 3.1 code files and instructions that follow. 
    - / Lab-3-1Links to an external site.
    - / Homework-3-1Links to an external site.

# Data Sets:

Use the /Homework-3-1Links to an external site. housing data and Google Maps data sets.

# Graded Exercise
The housing data set contains information about houses and their values, and the Google Maps raw data set contains information about addresses and their designations. Imagine we are building an ML tool to predict housing prices. To aid with prediction, we want to create a Neighborhood feature group. We can envision this neighborhood feature group helping us predict house prices by giving us a bucket to group new houses into.

## Feature Group:

After completing the above, generate the following features as part of their feature groups and screenshot the following queries against their feature store.

The neighborhood feature group should contain the following features:

- primary_key - neighborhood
    - derived from neighborhood-political
- event_time
    - time of ingestion to the feature store (calculated using Python)
- <1h ocean
    - one hot encoded column derived from ocean_proximity
- inland
    - one hot encoded column derived from ocean_proximity
- island
    - one hot encoded column derived from ocean_proximity
- near bay
    - one hot encoded column derived from ocean_proximity
- near ocean
    - one hot encoded column derived from ocean_proximity
- median house value
    - derived from median_house_value        
    - average this value across all records for a neighborhood        
    - cap this value at 500,000
- median house age        
    - derived from median_house_age        
    - average this value across all records for a neighborhood        
    - discretized by groups of 10 years i.e. 0-9, 10-19, 20-29, etc.
- total households        
    - derived from households        
    - average this value across all records for a neighborhood        
    - must be an integer (round up if needed)
- bedrooms per household        
    - derived from total_bedrooms and households
    - impute missing values by getting average for a postal-code 

# Query the Feature Values:

Please query the feature values from your feature store:

1. ) Brooktree

2. ) Fisherman’s Wharf

3. ) Los Osos

## Setup SageMaker FeatureStore

Let's start by setting up the SageMaker Python SDK and boto client. Note that this notebook requires a `boto3` version above `1.17.21`

In [41]:
import boto3
import sagemaker

original_boto3_version = boto3.__version__
%pip install "boto3>1.17.21"

Note: you may need to restart the kernel to use updated packages.


In [42]:
# from sagemaker.session import Session
from sagemaker.core.helper.session_helper import Session # Require for SageMaker v3 migration

region = boto3.Session().region_name

boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

#### S3 Bucket Setup For The OfflineStore

SageMaker FeatureStore writes the data in the OfflineStore of a FeatureGroup to a S3 bucket owned by you. To be able to write to your S3 bucket, SageMaker FeatureStore assumes an IAM role which has access to it. The role is also owned by you.
Note that the same bucket can be re-used across FeatureGroups. Data in the bucket is partitioned by FeatureGroup.

Set the default s3 bucket name and it will be referenced throughout the notebook.

In [43]:
# You can modify the following to use a bucket of your choosing
default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "sagemaker-featurestore-homwork3.1" # modify the bucket to sagemaker-featurestore-homwork3.1

bucket = default_s3_bucket_name

print(default_s3_bucket_name)


sagemaker-us-east-1-944202758041


Set up the IAM role. This role gives SageMaker FeatureStore access to your S3 bucket. 

<div class="alert alert-block alert-warning">
<b>Note:</b> In this example we use the default SageMaker role, assuming it has both <b>AmazonSageMakerFullAccess</b> and <b>AmazonSageMakerFeatureStoreAccess</b> managed policies. If not, please make sure to attach them to the role before proceeding.
</div>

In [44]:
# from sagemaker import get_execution_role
from sagemaker.core.helper.session_helper import get_execution_role # Update for SageMaker 3

# You can modify the following to use a role of your choosing. See the documentation for how to create this.
role = get_execution_role()
print(role)

arn:aws:iam::944202758041:role/LabRole


# Load the datasets

In [45]:
# Grab path to the dataset.csv

from pathlib import Path

current_directory = Path.cwd()

print("Current directory:")
print(current_directory)

print("\nCSV files found:")
csv_files = list(current_directory.rglob("*.csv"))

if not csv_files:
    print("No CSV files found.")
else:
    for file_path in csv_files:
        print(file_path.resolve())

Current directory:
/home/sagemaker-user/aai-540-homework/homework-3-1

CSV files found:
/home/sagemaker-user/aai-540-homework/homework-3-1/housing.csv
/home/sagemaker-user/aai-540-homework/homework-3-1/housing_gmaps_data_raw.csv


In [46]:
import pandas as pd
from pathlib import Path

housing_path = Path("/home/sagemaker-user/aai-540-homework/homework-3-1/housing.csv")
gmaps_path = Path("/home/sagemaker-user/aai-540-homework/homework-3-1/housing_gmaps_data_raw.csv")

for p in (housing_path, gmaps_path):
    if not p.is_file():
        raise FileNotFoundError(f"File not found: {p}")

df_housing = pd.read_csv(housing_path)
df_gmaps = pd.read_csv(gmaps_path)

# Exploratory Data Analysis

In [47]:
# Housing Dataset
print("Housing Dataset Information:")

# Dataset shape
print("Rows, Columns:", df_housing.shape)

# Dataset information
df_housing.info()

# Confirm no missing data remains
print("\nNull Count per Column (NaN Count):")
null_counts = df_housing.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.any() else "No missing values remain.")

# Check for duplicate data
duplicate_count = df_housing.duplicated().sum()

if duplicate_count > 0:
    print(f"\nFound {duplicate_count} duplicate rows!")
else:
    print("\nSuccess: No duplicate rows found!")

print("First 5 rows:")
df_housing.head()

Housing Dataset Information:
Rows, Columns: (20640, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB

Null Count per Column (NaN Count):
total_bedrooms    207
dtype: int64

Success: No duplicate rows found!
First 5 rows:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [48]:
# Maps Dataset
print("Maps Dataset Information:")

# Dataset shape
print("Rows, Columns:", df_gmaps.shape)

# Dataset information
df_gmaps.info()

# Confirm no missing data remains
print("\nNull Count per Column (NaN Count):")
null_counts = df_gmaps.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.any() else "No missing values remain.")

# Check for duplicate data
duplicate_count = df_gmaps.duplicated().sum()

if duplicate_count > 0:
    print(f"\nFound {duplicate_count} duplicate rows!")
else:
    print("\nSuccess: No duplicate rows found!")

print("First 5 rows:")
df_gmaps.head()

Maps Dataset Information:
Rows, Columns: (12590, 30)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12590 entries, 0 to 12589
Data columns (total 30 columns):
 #   Column                                                                              Non-Null Count  Dtype  
---  ------                                                                              --------------  -----  
 0   street_number                                                                       11188 non-null  object 
 1   route                                                                               12210 non-null  object 
 2   locality-political                                                                  12403 non-null  object 
 3   administrative_area_level_2-political                                               12543 non-null  object 
 4   administrative_area_level_1-political                                               12587 non-null  object 
 5   country-political                         

,street_number,route,locality-political,administrative_area_level_2-political,administrative_area_level_1-political,country-political,postal_code,address,longitude,latitude,...,establishment-natural_feature,airport-establishment-point_of_interest,political-sublocality-sublocality_level_1,administrative_area_level_3-political,post_box,establishment-light_rail_station-point_of_interest-transit_station,establishment-point_of_interest,aquarium-establishment-park-point_of_interest-tourist_attraction-zoo,campground-establishment-lodging-park-point_of_interest-rv_park-tourist_attraction,cemetery-establishment-park-point_of_interest
0,3130,Grizzly Peak Boulevard,Berkeley,Alameda County,California,United States,94705.0,"3130 Grizzly Peak Blvd, Berkeley, CA 94705, USA",-122.23,37.88,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005,Tunnel Road,Oakland,Alameda County,California,United States,94611.0,"2005 Tunnel Rd, Oakland, CA 94611, USA",-122.22,37.86,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6886,Chabot Road,Oakland,Alameda County,California,United States,94618.0,"6886 Chabot Rd, Oakland, CA 94618, USA",-122.24,37.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6365,Florio Street,Oakland,Alameda County,California,United States,94618.0,"6365 Florio St, Oakland, CA 94618, USA",-122.25,37.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5407,Bryant Avenue,Oakland,Alameda County,California,United States,94618.0,"5407 Bryant Ave, Oakland, CA 94618, USA",-122.25,37.84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Data Cleaning

In [49]:
# Housing Dataset

# Impute missing total_bedrooms with the median
df_housing['total_bedrooms'] = df_housing['total_bedrooms'].fillna(df_housing['total_bedrooms'].median())

# Confirm no missing data remains
print("\nNull Count per Column (NaN Count):")
null_counts = df_housing.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.any() else "No missing values remain.")


Null Count per Column (NaN Count):
No missing values remain.


In [50]:
# Maps Dataset

threshold = 0.90
null_frac = df_gmaps.isnull().mean()
cols_to_drop = null_frac[null_frac > threshold].index.tolist()
df_gmaps = df_gmaps.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns:", cols_to_drop)

# Keep neighborhood-political (needed for name lookups); drop only postal_code_suffix
df_gmaps = df_gmaps.drop(columns=['postal_code_suffix'], errors='ignore')

address_cols = ['street_number', 'route', 'locality-political',
                 'administrative_area_level_2-political', 'administrative_area_level_1-political',
                 'neighborhood-political']
present_cols = [c for c in address_cols if c in df_gmaps.columns]
df_gmaps[present_cols] = df_gmaps[present_cols].fillna("Unknown")

df_gmaps = df_gmaps.dropna(subset=['postal_code'])

print("\nRemaining nulls:", df_gmaps.isnull().sum().sum())
print("New shape:", df_gmaps.shape)

Dropped 18 columns: ['establishment-point_of_interest-transit_station', 'establishment-park-point_of_interest', 'premise', 'establishment-point_of_interest-subway_station-transit_station', 'airport-establishment-finance-moving_company-point_of_interest-storage', 'subpremise', 'bus_station-establishment-point_of_interest-transit_station', 'establishment-park-point_of_interest-tourist_attraction', 'establishment-natural_feature', 'airport-establishment-point_of_interest', 'political-sublocality-sublocality_level_1', 'administrative_area_level_3-political', 'post_box', 'establishment-light_rail_station-point_of_interest-transit_station', 'establishment-point_of_interest', 'aquarium-establishment-park-point_of_interest-tourist_attraction-zoo', 'campground-establishment-lodging-park-point_of_interest-rv_park-tourist_attraction', 'cemetery-establishment-park-point_of_interest']

Remaining nulls: 0
New shape: (12410, 11)


# Feature Engineering

In [51]:
# Build feature DataFrame

import pandas as pd
from time import time

df_neighborhood_fg = pd.DataFrame()

# Primary key: unique identifier per neighborhood record
df_neighborhood_fg["neighborhood_id"] = df_housing.index.astype(str)

# Event time, required by Feature Store (Unix epoch seconds)
df_neighborhood_fg["event_time"] = pd.Series([time()] * len(df_housing))

# One-hot encode ocean_proximity into individual binary features
ocean_dummies = pd.get_dummies(df_housing["ocean_proximity"], dtype="int64")
df_neighborhood_fg["near_1h_ocean"] = ocean_dummies.get("<1H OCEAN", 0)
df_neighborhood_fg["inland"] = ocean_dummies.get("INLAND", 0)
df_neighborhood_fg["island"] = ocean_dummies.get("ISLAND", 0)
df_neighborhood_fg["near_bay"] = ocean_dummies.get("NEAR BAY", 0)
df_neighborhood_fg["near_ocean"] = ocean_dummies.get("NEAR OCEAN", 0)

# Core numeric features
df_neighborhood_fg["median_house_value"] = df_housing["median_house_value"].astype("float64")
df_neighborhood_fg["median_house_age"] = df_housing["housing_median_age"].astype("float64")
df_neighborhood_fg["total_households"] = df_housing["households"].astype("int64")

# Engineered feature
df_neighborhood_fg["bedrooms_per_household"] = (
    df_housing["total_bedrooms"] / df_housing["households"]
).astype("float64")

print(df_neighborhood_fg.dtypes)
df_neighborhood_fg.head()

neighborhood_id            object
event_time                float64
near_1h_ocean               int64
inland                      int64
island                      int64
near_bay                    int64
near_ocean                  int64
median_house_value        float64
median_house_age          float64
total_households            int64
bedrooms_per_household    float64
dtype: object


,neighborhood_id,event_time,near_1h_ocean,inland,island,near_bay,near_ocean,median_house_value,median_house_age,total_households,bedrooms_per_household
0,0,1.789871e+09,0,0,0,1,0,452600.0,41.0,126,1.023810
1,1,1.789871e+09,0,0,0,1,0,358500.0,21.0,1138,0.971880
2,2,1.789871e+09,0,0,0,1,0,352100.0,52.0,177,1.073446
3,3,1.789871e+09,0,0,0,1,0,341300.0,52.0,219,1.073059
4,4,1.789871e+09,0,0,0,1,0,342200.0,52.0,259,1.081081


## Create the FeatureGroup in SageMaker Feature Store

In [52]:
import time
import boto3
from botocore.exceptions import ClientError

from sagemaker.core.resources import FeatureGroup
from sagemaker.core.shapes import (
    FeatureDefinition,
    OnlineStoreConfig,
    OfflineStoreConfig,
    S3StorageConfig,
)

record_identifier_feature_name = "neighborhood_id"
event_time_feature_name = "event_time"

# Map pandas dtypes to FeatureDefinitions manually (v3 SDK has no auto-detect helper)
dtype_to_feature_type = {
    "object": "String",
    "string": "String",
    "int64": "Integral",
    "int32": "Integral",
    "float64": "Fractional",
    "float32": "Fractional",
}

neighborhood_feature_definitions = [
    FeatureDefinition(
        feature_name=column,
        feature_type=dtype_to_feature_type.get(str(df_neighborhood_fg[column].dtype), "String"),
    )
    for column in df_neighborhood_fg.columns
]

offline_store_config = OfflineStoreConfig(
    s3_storage_config=S3StorageConfig(s3_uri=f"s3://{bucket}/{prefix}/neighborhood")
)

feature_group_name = "neighborhood-feature-group"  # hardcoded, stable name

# Check if it already exists, delete it if so
sagemaker_client = boto3.client("sagemaker")

try:
    sagemaker_client.describe_feature_group(FeatureGroupName=feature_group_name)
    exists = True
except ClientError as e:
    if "ResourceNotFound" in str(e):
        exists = False
    else:
        raise

if exists:
    print(f"FeatureGroup '{feature_group_name}' already exists. Deleting...")
    sagemaker_client.delete_feature_group(FeatureGroupName=feature_group_name)

    while True:
        try:
            sagemaker_client.describe_feature_group(FeatureGroupName=feature_group_name)
            print("Waiting for deletion...")
            time.sleep(5)
        except ClientError as e:
            if "ResourceNotFound" in str(e):
                print("FeatureGroup deleted.")
                break
            raise
else:
    print(f"FeatureGroup '{feature_group_name}' does not exist yet.")

neighborhood_feature_group = FeatureGroup.create(
    feature_group_name=feature_group_name,
    record_identifier_feature_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    feature_definitions=neighborhood_feature_definitions,
    role_arn=role,
    online_store_config=OnlineStoreConfig(enable_online_store=True),
    offline_store_config=offline_store_config,
)

neighborhood_feature_group.wait_for_status("Created")
print("Neighborhood FeatureGroup successfully created.")

FeatureGroup 'neighborhood-feature-group' already exists. Deleting...
Waiting for deletion...
FeatureGroup deleted.


[09/20/26 02:29:46] INFO     Creating feature_group resource.                                    ]8;id=6896146;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=6896147;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11769\11769]8;;\

Output()

[09/20/26 02:30:44] INFO     Final Resource Status: Created                                      ]8;id=6896152;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=6896153;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#12023\12023]8;;\

Neighborhood FeatureGroup successfully created.


# PutRecords into FeatureGroup

In [53]:
import boto3

featurestore_runtime = boto3.client("sagemaker-featurestore-runtime")

def ingest_dataframe(df, feature_group_name, client, log_every=1000):
    for i, row in df.iterrows():
        record = [
            {"FeatureName": col, "ValueAsString": str(row[col])}
            for col in df.columns
        ]
        client.put_record(FeatureGroupName=feature_group_name, Record=record)
        if (i + 1) % log_every == 0:
            print(f"Ingested {i + 1} / {len(df)} records")

ingest_dataframe(df_neighborhood_fg, feature_group_name, featurestore_runtime)
print("Ingestion complete.")

Ingested 1000 / 20640 records
Ingested 2000 / 20640 records
Ingested 3000 / 20640 records
Ingested 4000 / 20640 records
Ingested 5000 / 20640 records
Ingested 6000 / 20640 records
Ingested 7000 / 20640 records
Ingested 8000 / 20640 records
Ingested 9000 / 20640 records
Ingested 10000 / 20640 records
Ingested 11000 / 20640 records
Ingested 12000 / 20640 records
Ingested 13000 / 20640 records
Ingested 14000 / 20640 records
Ingested 15000 / 20640 records
Ingested 16000 / 20640 records
Ingested 17000 / 20640 records
Ingested 18000 / 20640 records
Ingested 19000 / 20640 records
Ingested 20000 / 20640 records
Ingestion complete.


## Confirm record is created successfully

In [54]:
import boto3

featurestore_runtime = boto3.client("sagemaker-featurestore-runtime")

sample_id = str(df_neighborhood_fg["neighborhood_id"].iloc[0])

# ---- Retrieve a record from the ONLINE store ----
online_record = featurestore_runtime.get_record(
    FeatureGroupName=feature_group_name,
    RecordIdentifierValueAsString=sample_id,
)
print(f"Online record for neighborhood_id = {sample_id}:")
for feature in online_record["Record"]:
    print(f"  {feature['FeatureName']}: {feature['ValueAsString']}")

# ---- Confirm OFFLINE store ingestion (writes to S3 within a few minutes of ingestion) ----
s3_client = boto3.client("s3")
offline_prefix = f"{prefix}/neighborhood"

response = s3_client.list_objects_v2(Bucket=bucket, Prefix=offline_prefix)
offline_files = [obj["Key"] for obj in response.get("Contents", [])]

if offline_files:
    print(f"\nFound {len(offline_files)} offline store objects in S3 (showing up to 5):")
    for key in offline_files[:5]:
        print(" ", key)
else:
    print("\nNo offline store objects found yet — offline writes can lag a few minutes behind ingestion. Wait and re-run this check.")

Online record for neighborhood_id = 0:
  neighborhood_id: 0
  event_time: 1789871374.6953528
  near_1h_ocean: 0
  inland: 0
  island: 0
  near_bay: 1
  near_ocean: 0
  median_house_value: 452600.0
  median_house_age: 41.0
  total_households: 126
  bedrooms_per_household: 1.0238095238095237

Found 172 offline store objects in S3 (showing up to 5):
  sagemaker-featurestore-homwork3.1/neighborhood/944202758041/sagemaker/us-east-1/offline-store/feature_group_name-1789863813/feature_group_name2026-09-20T00:23:33.491Z.txt
  sagemaker-featurestore-homwork3.1/neighborhood/944202758041/sagemaker/us-east-1/offline-store/neighborhood-feature-group-1789862347/data/year=2026/month=09/day=19/hour=23/20260919T235100Z_02335d9bedf07600.parquet
  sagemaker-featurestore-homwork3.1/neighborhood/944202758041/sagemaker/us-east-1/offline-store/neighborhood-feature-group-1789862347/data/year=2026/month=09/day=19/hour=23/20260919T235100Z_04b048d2f93ae08a.parquet
  sagemaker-featurestore-homwork3.1/neighborhood

# Query the Feature Values:

Please query the feature values from your feature store:

1.) Brooktree

2.) Fisherman’s Wharf

3.) Los Osos

In [55]:
target_neighborhoods = ["Brooktree", "Fisherman's Wharf", "Los Osos"]

matches = df_gmaps[df_gmaps["neighborhood-political"].isin(target_neighborhoods)]

for idx, name in zip(matches.index, matches["neighborhood-political"]):
    record = featurestore_runtime.get_record(
        FeatureGroupName=feature_group_name,
        RecordIdentifierValueAsString=str(idx),
    )
    print(f"\n{name} (neighborhood_id={idx}):")
    for feature in record.get("Record", []):
        print(f"  {feature['FeatureName']}: {feature['ValueAsString']}")


Fisherman's Wharf (neighborhood_id=9252):
  neighborhood_id: 9252
  event_time: 1789871374.6953528
  near_1h_ocean: 0
  inland: 1
  island: 0
  near_bay: 0
  near_ocean: 0
  median_house_value: 73800.0
  median_house_age: 36.0
  total_households: 363
  bedrooms_per_household: 0.9614325068870524

Los Osos (neighborhood_id=9689):
  neighborhood_id: 9689
  event_time: 1789871374.6953528
  near_1h_ocean: 1
  inland: 0
  island: 0
  near_bay: 0
  near_ocean: 0
  median_house_value: 148100.0
  median_house_age: 29.0
  total_households: 1036
  bedrooms_per_household: 1.0193050193050193

Los Osos (neighborhood_id=9690):
  neighborhood_id: 9690
  event_time: 1789871374.6953528
  near_1h_ocean: 1
  inland: 0
  island: 0
  near_bay: 0
  near_ocean: 0
  median_house_value: 157500.0
  median_house_age: 27.0
  total_households: 815
  bedrooms_per_household: 1.0871165644171779

Los Osos (neighborhood_id=9691):
  neighborhood_id: 9691
  event_time: 1789871374.6953528
  near_1h_ocean: 1
  inland: 0
  